In [1]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from utils.experiment_separator import get_experiments, separate_experiments

In [2]:
data_path = "../data/data.csv"
column_names = ["density", "cutting_speed", "feed_rate", "depth", "axial_force", "cutting_force"]
target = "cutting_force"
input_columns = ["density", "cutting_speed", "feed_rate", "depth", "axial_force"]

depth_quantile = 0.8
window_size = 100
stride = 5
hidden_size = 128
num_layers = 1
dropout = 0.0
epochs = 30
batch_size = 64
lr = 1e-3
patience = 5
min_delta = 0.0
scheduler_factor = 0.5
scheduler_patience = 2
val_fraction = 0.15
seed = 42


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [3]:
lstm_config = {
    "window_size": 100, "stride": 5, "hidden_size": 128,
    "num_layers": 1, "batch_size": 64, "lr": 1e-3,
    "dropout": 0.0, "optimizer": "Adam", "weight_decay": 0.0,
    "gradient_clip": None,
}
gru_config = {
    "window_size": 75, "stride": 10, "hidden_size": 128,
    "num_layers": 2, "batch_size": 128, "lr": 2e-3,
    "dropout": 0.3, "optimizer": "Adam", "weight_decay": 0.0,
    "gradient_clip": None,
}
rnn_config = {
    "window_size": 100, "stride": 5, "hidden_size": 128,
    "num_layers": 3, "batch_size": 128, "lr": 1e-3,
    "dropout": 0.0, "optimizer": "Adam", "weight_decay": 0.0,
    "gradient_clip": None,
}
tcn_config = {
    "window_size": 76, "stride": 10, "channels": [64] * 5,
    "kernel_size": 3, "batch_size": 128, "lr": 1e-3,
    "dropout": 0.1, "optimizer": "AdamW", "weight_decay": 1e-4,
    "gradient_clip": 1.0,
}

In [4]:
df = pd.read_csv(data_path)
df.columns = column_names
df = separate_experiments(df)
experiments = get_experiments(df)

depth_threshold = df["depth"].quantile(depth_quantile)
train_mask = df["depth"] < depth_threshold
test_mask = ~train_mask

print(f"Depth threshold ({depth_quantile:.0%}): {depth_threshold:.4f}")
print(f"Rows below threshold (train): {train_mask.sum():,}")
print(f"Rows at/above threshold (test): {test_mask.sum():,}")

Depth threshold (80%): 26.3864
Rows below threshold (train): 226,512
Rows at/above threshold (test): 56,628


In [5]:
X_train_cls = df.loc[train_mask, input_columns].values
y_train_cls = df.loc[train_mask, target].values
X_test_cls = df.loc[test_mask, input_columns].values
y_test_cls = df.loc[test_mask, target].values

linear_model = LinearRegression()
linear_model.fit(X_train_cls, y_train_cls)
linear_preds = linear_model.predict(X_test_cls)

hgb_model = HistGradientBoostingRegressor(random_state=seed)
hgb_model.fit(X_train_cls, y_train_cls)
hgb_preds = hgb_model.predict(X_test_cls)

classical_results = []
for name, preds in [("LinearRegression", linear_preds), ("HistGradientBoosting", hgb_preds)]:
    mse = mean_squared_error(y_test_cls, preds)
    classical_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test_cls, preds),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_test_cls, preds),
    })
    print(f"{name}: RMSE={np.sqrt(mse):.4f} | R2={r2_score(y_test_cls, preds):.4f}")

LinearRegression: RMSE=191.3532 | R2=0.6064
HistGradientBoosting: RMSE=68.6607 | R2=0.9493


In [6]:
scale_cols = input_columns + [target]
scaler = StandardScaler()
scaler.fit(df.loc[train_mask, scale_cols])

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [7]:
set_seed(seed)


def build_windows(window_size, stride):
    train_windows, test_windows, win_test_depths = [], [], []

    for key, exp_df in experiments.items():
        if len(exp_df) < window_size:
            continue
        depth_vals = exp_df["depth"].values
        scaled_vals = scaler.transform(exp_df[scale_cols])
        input_vals = scaled_vals[:, :-1]
        target_vals = scaled_vals[:, -1]

        for i in range(0, len(exp_df) - window_size + 1, stride):
            end = i + window_size - 1
            sample = (input_vals[i:i + window_size].copy(), target_vals[end])
            if depth_vals[end] < depth_threshold:
                train_windows.append(sample)
            else:
                test_windows.append(sample)
                win_test_depths.append(depth_vals[end])

    rng = np.random.default_rng(seed)
    val_size = int(len(train_windows) * val_fraction)
    val_idx = set(rng.choice(len(train_windows), size=val_size, replace=False))

    train_only = [w for i, w in enumerate(train_windows) if i not in val_idx]
    val_only = [w for i, w in enumerate(train_windows) if i in val_idx]

    def to_tensors(windows):
        X = torch.tensor(np.array([w[0] for w in windows]), dtype=torch.float32)
        y = torch.tensor(np.array([w[1] for w in windows]), dtype=torch.float32).unsqueeze(1)
        return X, y

    X_tr, y_tr = to_tensors(train_only)
    X_va, y_va = to_tensors(val_only)
    X_te, y_te = to_tensors(test_windows)
    return X_tr, y_tr, X_va, y_va, X_te, y_te, np.array(win_test_depths)


X_train, y_train, X_val, y_val, X_test, y_test, test_depths = build_windows(window_size, stride)
print(f"Train windows: {len(X_train):,} | Val windows: {len(X_val):,} | Test windows: {len(X_test):,}")

Train windows: 35,231 | Val windows: 6,217 | Test windows: 11,418


In [8]:
class GRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size + input_size, 1)

    def forward(self, x):
        _, h_n = self.gru(x)
        combined = torch.cat([h_n[-1], x[:, -1, :]], dim=1)
        return self.fc(combined)


class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size + input_size, 1)

    def forward(self, x):
        _, h_n = self.rnn(x)
        combined = torch.cat([h_n[-1], x[:, -1, :]], dim=1)
        return self.fc(combined)


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.padding = padding
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.dropout = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def _causal_crop(self, x):
        return x[:, :, :-self.padding] if self.padding else x

    def forward(self, x):
        out = self._causal_crop(self.conv1(x))
        out = self.dropout(torch.relu(out))
        out = self._causal_crop(self.conv2(out))
        out = self.dropout(torch.relu(out))
        return torch.relu(out + self.residual(x))


class TCN(nn.Module):
    def __init__(self, input_size, channels, kernel_size, dropout):
        super().__init__()
        layers = []
        in_channels = input_size
        for index, out_channels in enumerate(channels):
            layers.append(TemporalBlock(in_channels, out_channels, kernel_size, 2 ** index, dropout))
            in_channels = out_channels
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(channels[-1] + input_size, 1)

    def forward(self, x):
        features = self.network(x.transpose(1, 2))[:, :, -1]
        return self.fc(torch.cat([features, x[:, -1, :]], dim=1))


def train_model(model, X_train, y_train, X_val, y_val, batch_size, lr, optimizer_name, weight_decay, gradient_clip):
    criterion = nn.SmoothL1Loss(beta=1.0)
    val_criterion = nn.MSELoss()
    optimizer = getattr(torch.optim, optimizer_name)(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=scheduler_factor, patience=scheduler_patience
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, generator=generator)

    best_val_loss = float("inf")
    best_state_dict = None
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            if gradient_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
            optimizer.step()
            train_loss += loss.item()

        val_loss = calculate_loss_in_batches(model, X_val, y_val, val_criterion, batch_size)
        scheduler.step(val_loss)
        train_loss /= len(train_loader)
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch + 1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.2e}")

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    model.load_state_dict(best_state_dict)
    return model

In [9]:
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size + input_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        combined = torch.cat([h_n[-1], x[:, -1, :]], dim=1)
        return self.fc(combined)


def calculate_loss_in_batches(model, X, y, criterion, batch_size):
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=False)
    total_loss, total_samples = 0.0, 0
    model.eval()
    with torch.no_grad():
        for X_batch, y_batch in loader:
            loss = criterion(model(X_batch), y_batch)
            total_loss += loss.item() * X_batch.size(0)
            total_samples += X_batch.size(0)
    return total_loss / total_samples

In [10]:
model = LSTM(len(input_columns), hidden_size, num_layers, dropout)
criterion = nn.SmoothL1Loss(beta=1.0)
val_criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=scheduler_factor, patience=scheduler_patience
)
generator = torch.Generator().manual_seed(seed)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, generator=generator)

best_val_loss = float("inf")
best_state_dict = None
epochs_without_improvement = 0

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    val_loss = calculate_loss_in_batches(model, X_val, y_val, val_criterion, batch_size)
    scheduler.step(val_loss)
    train_loss /= len(train_loader)
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch + 1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.2e}")

    if val_loss < best_val_loss - min_delta:
        best_val_loss = val_loss
        best_state_dict = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

model.load_state_dict(best_state_dict)

Epoch 1/30 | Train Loss: 0.0139 | Val Loss: 0.0054 | LR: 1.00e-03
Epoch 2/30 | Train Loss: 0.0022 | Val Loss: 0.0032 | LR: 1.00e-03
Epoch 3/30 | Train Loss: 0.0021 | Val Loss: 0.0045 | LR: 1.00e-03
Epoch 4/30 | Train Loss: 0.0018 | Val Loss: 0.0029 | LR: 1.00e-03
Epoch 5/30 | Train Loss: 0.0017 | Val Loss: 0.0032 | LR: 1.00e-03
Epoch 6/30 | Train Loss: 0.0015 | Val Loss: 0.0049 | LR: 1.00e-03
Epoch 7/30 | Train Loss: 0.0013 | Val Loss: 0.0023 | LR: 1.00e-03
Epoch 8/30 | Train Loss: 0.0013 | Val Loss: 0.0021 | LR: 1.00e-03
Epoch 9/30 | Train Loss: 0.0013 | Val Loss: 0.0025 | LR: 1.00e-03
Epoch 10/30 | Train Loss: 0.0013 | Val Loss: 0.0018 | LR: 1.00e-03
Epoch 11/30 | Train Loss: 0.0011 | Val Loss: 0.0015 | LR: 1.00e-03
Epoch 12/30 | Train Loss: 0.0009 | Val Loss: 0.0014 | LR: 1.00e-03
Epoch 13/30 | Train Loss: 0.0011 | Val Loss: 0.0018 | LR: 1.00e-03
Epoch 14/30 | Train Loss: 0.0011 | Val Loss: 0.0013 | LR: 1.00e-03
Epoch 15/30 | Train Loss: 0.0007 | Val Loss: 0.0015 | LR: 1.00e-03
Epoc

<All keys matched successfully>

In [11]:
def inverse_transform_target(scaler, values):
    n_cols = len(input_columns) + 1
    dummy = np.zeros((len(values), n_cols))
    dummy[:, -1] = values
    return scaler.inverse_transform(dummy)[:, -1]


def predict_scaled(model, X, batch_size):
    loader = DataLoader(X, batch_size=batch_size, shuffle=False)
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X_batch) for X_batch in loader]).squeeze().numpy()


lstm_preds_scaled = predict_scaled(model, X_test, batch_size)
lstm_preds = inverse_transform_target(scaler, lstm_preds_scaled)
lstm_actuals = inverse_transform_target(scaler, y_test.squeeze().numpy())

mse = mean_squared_error(lstm_actuals, lstm_preds)
lstm_result = {
    "Model": "LSTM",
    "MAE": mean_absolute_error(lstm_actuals, lstm_preds),
    "MSE": mse,
    "RMSE": np.sqrt(mse),
    "R2": r2_score(lstm_actuals, lstm_preds),
}
print(f"LSTM: RMSE={lstm_result['RMSE']:.4f} | R2={lstm_result['R2']:.4f}")

LSTM: RMSE=42.6193 | R2=0.9805


In [12]:
set_seed(seed)
gru_X_train, gru_y_train, gru_X_val, gru_y_val, gru_X_test, gru_y_test, gru_test_depths = build_windows(
    gru_config["window_size"], gru_config["stride"]
)
print(f"Train windows: {len(gru_X_train):,} | Val windows: {len(gru_X_val):,} | Test windows: {len(gru_X_test):,}")

gru_model = GRU(len(input_columns), gru_config["hidden_size"], gru_config["num_layers"], gru_config["dropout"])
gru_model = train_model(
    gru_model, gru_X_train, gru_y_train, gru_X_val, gru_y_val,
    gru_config["batch_size"], gru_config["lr"], gru_config["optimizer"],
    gru_config["weight_decay"], gru_config["gradient_clip"],
)

gru_preds_scaled = predict_scaled(gru_model, gru_X_test, gru_config["batch_size"])
gru_preds = inverse_transform_target(scaler, gru_preds_scaled)
gru_actuals = inverse_transform_target(scaler, gru_y_test.squeeze().numpy())

mse = mean_squared_error(gru_actuals, gru_preds)
gru_result = {
    "Model": "GRU",
    "MAE": mean_absolute_error(gru_actuals, gru_preds),
    "MSE": mse,
    "RMSE": np.sqrt(mse),
    "R2": r2_score(gru_actuals, gru_preds),
}
print(f"GRU: RMSE={gru_result['RMSE']:.4f} | R2={gru_result['R2']:.4f}")

Train windows: 18,065 | Val windows: 3,187 | Test windows: 5,676
Epoch 1/30 | Train Loss: 0.0306 | Val Loss: 0.0116 | LR: 2.00e-03
Epoch 2/30 | Train Loss: 0.0039 | Val Loss: 0.0047 | LR: 2.00e-03
Epoch 3/30 | Train Loss: 0.0033 | Val Loss: 0.0046 | LR: 2.00e-03
Epoch 4/30 | Train Loss: 0.0023 | Val Loss: 0.0039 | LR: 2.00e-03
Epoch 5/30 | Train Loss: 0.0024 | Val Loss: 0.0031 | LR: 2.00e-03
Epoch 6/30 | Train Loss: 0.0022 | Val Loss: 0.0046 | LR: 2.00e-03
Epoch 7/30 | Train Loss: 0.0019 | Val Loss: 0.0075 | LR: 2.00e-03
Epoch 8/30 | Train Loss: 0.0025 | Val Loss: 0.0027 | LR: 2.00e-03
Epoch 9/30 | Train Loss: 0.0020 | Val Loss: 0.0032 | LR: 2.00e-03
Epoch 10/30 | Train Loss: 0.0016 | Val Loss: 0.0086 | LR: 2.00e-03
Epoch 11/30 | Train Loss: 0.0018 | Val Loss: 0.0044 | LR: 1.00e-03
Epoch 12/30 | Train Loss: 0.0014 | Val Loss: 0.0025 | LR: 1.00e-03
Epoch 13/30 | Train Loss: 0.0013 | Val Loss: 0.0021 | LR: 1.00e-03
Epoch 14/30 | Train Loss: 0.0013 | Val Loss: 0.0022 | LR: 1.00e-03
Epoch 

In [13]:
set_seed(seed)
rnn_X_train, rnn_y_train, rnn_X_val, rnn_y_val, rnn_X_test, rnn_y_test, rnn_test_depths = build_windows(
    rnn_config["window_size"], rnn_config["stride"]
)
print(f"Train windows: {len(rnn_X_train):,} | Val windows: {len(rnn_X_val):,} | Test windows: {len(rnn_X_test):,}")

rnn_model = RNN(len(input_columns), rnn_config["hidden_size"], rnn_config["num_layers"], rnn_config["dropout"])
rnn_model = train_model(
    rnn_model, rnn_X_train, rnn_y_train, rnn_X_val, rnn_y_val,
    rnn_config["batch_size"], rnn_config["lr"], rnn_config["optimizer"],
    rnn_config["weight_decay"], rnn_config["gradient_clip"],
)

rnn_preds_scaled = predict_scaled(rnn_model, rnn_X_test, rnn_config["batch_size"])
rnn_preds = inverse_transform_target(scaler, rnn_preds_scaled)
rnn_actuals = inverse_transform_target(scaler, rnn_y_test.squeeze().numpy())

mse = mean_squared_error(rnn_actuals, rnn_preds)
rnn_result = {
    "Model": "RNN",
    "MAE": mean_absolute_error(rnn_actuals, rnn_preds),
    "MSE": mse,
    "RMSE": np.sqrt(mse),
    "R2": r2_score(rnn_actuals, rnn_preds),
}
print(f"RNN: RMSE={rnn_result['RMSE']:.4f} | R2={rnn_result['R2']:.4f}")

Train windows: 35,231 | Val windows: 6,217 | Test windows: 11,418
Epoch 1/30 | Train Loss: 0.0175 | Val Loss: 0.0069 | LR: 1.00e-03
Epoch 2/30 | Train Loss: 0.0045 | Val Loss: 0.0062 | LR: 1.00e-03
Epoch 3/30 | Train Loss: 0.0027 | Val Loss: 0.0055 | LR: 1.00e-03
Epoch 4/30 | Train Loss: 0.0030 | Val Loss: 0.0069 | LR: 1.00e-03
Epoch 5/30 | Train Loss: 0.0023 | Val Loss: 0.0036 | LR: 1.00e-03
Epoch 6/30 | Train Loss: 0.0022 | Val Loss: 0.0046 | LR: 1.00e-03
Epoch 7/30 | Train Loss: 0.0022 | Val Loss: 0.0041 | LR: 1.00e-03
Epoch 8/30 | Train Loss: 0.0025 | Val Loss: 0.0046 | LR: 5.00e-04
Epoch 9/30 | Train Loss: 0.0015 | Val Loss: 0.0037 | LR: 5.00e-04
Epoch 10/30 | Train Loss: 0.0016 | Val Loss: 0.0025 | LR: 5.00e-04
Epoch 11/30 | Train Loss: 0.0013 | Val Loss: 0.0034 | LR: 5.00e-04
Epoch 12/30 | Train Loss: 0.0014 | Val Loss: 0.0021 | LR: 5.00e-04
Epoch 13/30 | Train Loss: 0.0013 | Val Loss: 0.0030 | LR: 5.00e-04
Epoch 14/30 | Train Loss: 0.0012 | Val Loss: 0.0019 | LR: 5.00e-04
Epoch

In [14]:
set_seed(seed)
tcn_X_train, tcn_y_train, tcn_X_val, tcn_y_val, tcn_X_test, tcn_y_test, tcn_test_depths = build_windows(
    tcn_config["window_size"], tcn_config["stride"]
)
print(f"Train windows: {len(tcn_X_train):,} | Val windows: {len(tcn_X_val):,} | Test windows: {len(tcn_X_test):,}")

tcn_model = TCN(len(input_columns), tcn_config["channels"], tcn_config["kernel_size"], tcn_config["dropout"])
tcn_model = train_model(
    tcn_model, tcn_X_train, tcn_y_train, tcn_X_val, tcn_y_val,
    tcn_config["batch_size"], tcn_config["lr"], tcn_config["optimizer"],
    tcn_config["weight_decay"], tcn_config["gradient_clip"],
)

tcn_preds_scaled = predict_scaled(tcn_model, tcn_X_test, tcn_config["batch_size"])
tcn_preds = inverse_transform_target(scaler, tcn_preds_scaled)
tcn_actuals = inverse_transform_target(scaler, tcn_y_test.squeeze().numpy())

mse = mean_squared_error(tcn_actuals, tcn_preds)
tcn_result = {
    "Model": "TCN",
    "MAE": mean_absolute_error(tcn_actuals, tcn_preds),
    "MSE": mse,
    "RMSE": np.sqrt(mse),
    "R2": r2_score(tcn_actuals, tcn_preds),
}
print(f"TCN: RMSE={tcn_result['RMSE']:.4f} | R2={tcn_result['R2']:.4f}")

Train windows: 18,065 | Val windows: 3,187 | Test windows: 5,676
Epoch 1/30 | Train Loss: 0.0244 | Val Loss: 0.0088 | LR: 1.00e-03
Epoch 2/30 | Train Loss: 0.0046 | Val Loss: 0.0035 | LR: 1.00e-03
Epoch 3/30 | Train Loss: 0.0044 | Val Loss: 0.0031 | LR: 1.00e-03
Epoch 4/30 | Train Loss: 0.0031 | Val Loss: 0.0040 | LR: 1.00e-03
Epoch 5/30 | Train Loss: 0.0025 | Val Loss: 0.0033 | LR: 1.00e-03
Epoch 6/30 | Train Loss: 0.0026 | Val Loss: 0.0042 | LR: 5.00e-04
Epoch 7/30 | Train Loss: 0.0019 | Val Loss: 0.0022 | LR: 5.00e-04
Epoch 8/30 | Train Loss: 0.0018 | Val Loss: 0.0018 | LR: 5.00e-04
Epoch 9/30 | Train Loss: 0.0017 | Val Loss: 0.0022 | LR: 5.00e-04
Epoch 10/30 | Train Loss: 0.0016 | Val Loss: 0.0028 | LR: 5.00e-04
Epoch 11/30 | Train Loss: 0.0014 | Val Loss: 0.0022 | LR: 2.50e-04
Epoch 12/30 | Train Loss: 0.0013 | Val Loss: 0.0017 | LR: 2.50e-04
Epoch 13/30 | Train Loss: 0.0013 | Val Loss: 0.0020 | LR: 2.50e-04
Epoch 14/30 | Train Loss: 0.0013 | Val Loss: 0.0018 | LR: 2.50e-04
Epoch 

In [15]:
comparison = pd.DataFrame(classical_results + [lstm_result, gru_result, rnn_result, tcn_result]).set_index("Model")
display(comparison.round(4))

,MAE,MSE,RMSE,R2
Model,,,,
LinearRegression,136.0979,36616.0567,191.3532,0.6064
HistGradientBoosting,43.3371,4714.2959,68.6607,0.9493
LSTM,25.4819,1816.4033,42.6193,0.9805
GRU,35.3124,3860.9530,62.1366,0.9584
RNN,38.3521,4409.3077,66.4026,0.9526
TCN,26.1476,1694.2628,41.1614,0.9818


In [16]:
def bucket_errors(depths, actuals, preds, threshold, n_buckets=5):
    bins = np.linspace(threshold, depths.max(), n_buckets + 1)
    bucket_idx = np.clip(np.digitize(depths, bins) - 1, 0, n_buckets - 1)
    rows = []
    for b in range(n_buckets):
        mask = bucket_idx == b
        if mask.sum() == 0:
            continue
        mse = mean_squared_error(actuals[mask], preds[mask])
        rows.append({
            "Depth Range": f"{bins[b]:.2f}-{bins[b + 1]:.2f}",
            "N": int(mask.sum()),
            "RMSE": np.sqrt(mse),
        })
    return pd.DataFrame(rows)


cls_test_depths = df.loc[test_mask, "depth"].values

print("LinearRegression by depth bucket (distance past threshold):")
display(bucket_errors(cls_test_depths, y_test_cls, linear_preds, depth_threshold).round(4))

print("\nHistGradientBoosting by depth bucket (distance past threshold):")
display(bucket_errors(cls_test_depths, y_test_cls, hgb_preds, depth_threshold).round(4))

print("\nLSTM by depth bucket (distance past threshold):")
display(bucket_errors(test_depths, lstm_actuals, lstm_preds, depth_threshold).round(4))

print("\nGRU by depth bucket (distance past threshold):")
display(bucket_errors(gru_test_depths, gru_actuals, gru_preds, depth_threshold).round(4))

print("\nRNN by depth bucket (distance past threshold):")
display(bucket_errors(rnn_test_depths, rnn_actuals, rnn_preds, depth_threshold).round(4))

print("\nTCN by depth bucket (distance past threshold):")
display(bucket_errors(tcn_test_depths, tcn_actuals, tcn_preds, depth_threshold).round(4))

LinearRegression by depth bucket (distance past threshold):


,Depth Range,N,RMSE
0,26.39-27.71,11352,162.4914
1,27.71-29.03,11286,183.2984
2,29.03-30.34,11286,198.3808
3,30.34-31.66,11286,204.9116
4,31.66-32.98,11418,204.2907



HistGradientBoosting by depth bucket (distance past threshold):


,Depth Range,N,RMSE
0,26.39-27.71,11352,25.0031
1,27.71-29.03,11286,50.6996
2,29.03-30.34,11286,73.1425
3,30.34-31.66,11286,85.1440
4,31.66-32.98,11418,88.1180



LSTM by depth bucket (distance past threshold):


...

,Depth Range,N,RMSE
0,26.39-27.69,1122,12.6020
1,27.69-28.99,1122,25.0205
2,28.99-30.29,1122,45.5765
3,30.29-31.60,1122,72.2886
4,31.60-32.90,1188,103.9634



RNN by depth bucket (distance past threshold):


,Depth Range,N,RMSE
0,26.39-27.71,2244,10.2930
1,27.71-29.03,2310,22.6627
2,29.03-30.34,2244,61.2408
3,30.34-31.66,2310,78.4997
4,31.66-32.98,2310,106.6423



TCN by depth bucket (distance past threshold):


,Depth Range,N,RMSE
0,26.39-27.69,1122,9.4006
1,27.69-29.00,1122,17.3515
2,29.00-30.30,1122,30.1135
3,30.30-31.61,1122,47.0621
4,31.61-32.92,1188,69.1286
